<a href="https://colab.research.google.com/github/jeevaprakash-S/24ADI006_24BAD048/blob/main/Experiment_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q transformers datasets accelerate evaluate scikit-learn torch

# **PART A: DATASET PREPARATION AND TEXT PREPROCESSING**

In [1]:
print("24BAD048 - Jeevaprakash S")

# Import Libraries
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

# Load IMDb Dataset
dataset = load_dataset("stanfordnlp/imdb")

print("\nDataset Loaded Successfully!")
print(dataset)

# Explore Dataset
print("\nTraining Samples:", len(dataset["train"]))
print("Testing Samples:", len(dataset["test"]))

# Display Sample Reviews
print("\nSample Dataset Entries:")
for i in range(5):
    print("\nReview:", dataset["train"][i]["text"][:300])
    print("Label:", dataset["train"][i]["label"])

# Check Class Distribution
df = pd.DataFrame(dataset["train"])
label_counts = df["label"].value_counts().sort_index()

print("\nClass Distribution:")
print("Negative Reviews (0):", label_counts[0])
print("Positive Reviews (1):", label_counts[1])

# Prepare Training and Testing Data
train_dataset = dataset["train"].shuffle(seed=42).select(range(5000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(1000))

print("\nTraining Samples:", len(train_dataset))
print("Testing Samples:", len(test_dataset))

# Load Pretrained BERT Tokenizer
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\nBERT Tokenizer Loaded Successfully!")

# Tokenize Sample Text
sample_text = train_dataset[0]["text"]

tokens = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=256
)

print("\nOriginal Text:")
print(sample_text[:300])

print("\nTokenized Input IDs:")
print(tokens["input_ids"][:30])

print("\nAttention Mask:")
print(tokens["attention_mask"][:30])

# Tokenization Function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

# Tokenize Training and Testing Datasets
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

# Remove Text Column
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

# Rename Label Column
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

# Convert into PyTorch Format
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

# Prepare Batch Processing
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("\nTokenization Completed Successfully!")
print("Training Dataset:", len(tokenized_train))
print("Testing Dataset:", len(tokenized_test))
print("Dataset prepared for batch-wise processing.")

print("\nPART A COMPLETED SUCCESSFULLY!")

24BAD048 - Jeevaprakash S


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (10 outputs left out)


BERT Tokenizer Loaded Successfully!

Original Text:
There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot 

Tokenized Input IDs:
[101, 2045, 2003, 2053, 7189, 2012, 2035, 2090, 3481, 3771, 1998, 6337, 2099, 2021, 1996, 2755, 2008, 2119, 2024, 2610, 2186, 2055, 6355, 6997, 1012, 6337, 2099, 3504, 15594, 2100]

Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


Tokenization Completed Successfully!
Training Dataset: 5000
Testing Dataset: 1000
Dataset prepared for batch-wise processing.

PART A COMPLETED SUCCESSFULLY!


# **PART B: FINE-TUNING BERT FOR TEXT CLASSIFICATION**

In [4]:
print("24BAD048 - Jeevaprakash S")

# Import Libraries
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

from datasets import load_dataset

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing Device:", device)

# Load IMDb Dataset
dataset = load_dataset("stanfordnlp/imdb")

train_dataset = dataset["train"].shuffle(seed=42).select(range(5000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(1000))

# Load BERT Tokenizer
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Tokenization
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Load Pretrained BERT Model
bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

bert_model.to(device)

print("\nBERT Model Loaded Successfully!")

# Define Evaluation Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, average="weighted"),
        "recall": recall_score(labels, predictions, average="weighted"),
        "f1": f1_score(labels, predictions, average="weighted")
    }

# Training Parameters
training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    fp16=torch.cuda.is_available()
)

# Initialize Trainer
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Fine-Tune BERT
print("\nStarting BERT Training...")
trainer.train()

print("\nBERT Training Completed!")

# Evaluate Model
evaluation_results = trainer.evaluate()

print("\nBERT Evaluation Results:")
for metric, value in evaluation_results.items():
    if isinstance(value, (int, float)):
        print(f"{metric}: {value:.4f}")

# Classification Report
predictions_output = trainer.predict(tokenized_test)

predictions = np.argmax(predictions_output.predictions, axis=1)
actual_labels = predictions_output.label_ids

print("\nClassification Report:")
print(classification_report(
    actual_labels,
    predictions,
    target_names=["Negative", "Positive"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(actual_labels, predictions))

# Test New Text Inputs
def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    bert_model.eval()

    with torch.no_grad():
        outputs = bert_model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()

    label = "Positive" if predicted_class == 1 else "Negative"
    confidence = probabilities[0][predicted_class].item()

    print("\nReview:", text)
    print("Predicted Sentiment:", label)
    print("Confidence:", round(confidence * 100, 2), "%")

    return label

# Test Positive Review
predict_sentiment(
    "This movie was fantastic! The acting and storyline were amazing."
)

# Test Negative Review
predict_sentiment(
    "The movie was boring and disappointing. I did not enjoy it."
)

# Save Model and Tokenizer
save_path = f"./24BAD048_BERT_Model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("\nBERT Model Saved Successfully!")
print("Location:", save_path)

print("\nPART B COMPLETED SUCCESSFULLY!")

24BAD048 - Jeevaprakash S

Using Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



BERT Model Loaded Successfully!

Starting BERT Training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.350099,0.356321,0.883000,0.889102,0.883000,0.882736
2,0.162315,0.392638,0.889000,0.890465,0.889000,0.888984


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)


BERT Evaluation Results:
eval_loss: 0.3926
eval_accuracy: 0.8890
eval_precision: 0.8905
eval_recall: 0.8890
eval_f1: 0.8890



Classification Report:
              precision    recall  f1-score   support

    Negative       0.92      0.86      0.89       512
    Positive       0.86      0.92      0.89       488

    accuracy                           0.89      1000
   macro avg       0.89      0.89      0.89      1000
weighted avg       0.89      0.89      0.89      1000


Confusion Matrix:
[[442  70]
 [ 41 447]]

Review: This movie was fantastic! The acting and storyline were amazing.
Predicted Sentiment: Positive
Confidence: 99.72 %

Review: The movie was boring and disappointing. I did not enjoy it.
Predicted Sentiment: Negative
Confidence: 99.64 %


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


BERT Model Saved Successfully!
Location: ./24BAD048_BERT_Model

PART B COMPLETED SUCCESSFULLY!


In [5]:
print("24BAD048 - Jeevaprakash S")

# Import Libraries
import torch
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing Device:", device)

# Load T5-small Tokenizer
T5_MODEL_NAME = "t5-small"

t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL_NAME)

# Load T5 Model
t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL_NAME)

t5_model.to(device)

print("\nT5-small Model Loaded Successfully!")

# Prepare Sample Input Text
input_text = """
The movie tells the story of a young man who discovers
his passion for music. Despite facing many challenges,
he works hard to achieve his dreams. The actors deliver
excellent performances and the background music makes
the movie more emotional and enjoyable.
"""

print("\nOriginal Input Text:")
print(input_text)

# Preprocess Input
if not input_text.strip().lower().startswith("summarize:"):
    input_text = "summarize: " + input_text

inputs = t5_tokenizer(
    input_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(device)

print("\nInput Preprocessing Completed!")
print("Input IDs Shape:", inputs["input_ids"].shape)

# Generate Summary
t5_model.eval()

with torch.no_grad():
    output_ids = t5_model.generate(
        inputs["input_ids"],
        max_length=100,
        min_length=20,
        num_beams=4,
        early_stopping=True
    )

generated_summary = t5_tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("\nGenerated Summary:")
print(generated_summary)

# Function to Generate Summary
def generate_summary(text, num_beams, max_length):

    if not text.strip().lower().startswith("summarize:"):
        text = "summarize: " + text

    inputs = t5_tokenizer(
        text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = t5_model.generate(
            inputs["input_ids"],
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=True
        )

    return t5_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

# Experiment with Different Inputs and Settings
sample_review = """
The film is about a group of friends who travel to a
beautiful island. They experience many adventures and
discover the importance of friendship, teamwork and courage.
"""

summary_1 = generate_summary(sample_review, 1, 60)
summary_2 = generate_summary(sample_review, 4, 60)
summary_3 = generate_summary(sample_review, 6, 100)

print("\nOriginal Text:")
print(sample_review)

print("\nBeam Search = 1:")
print(summary_1)

print("\nBeam Search = 4:")
print(summary_2)

print("\nBeam Search = 6:")
print(summary_3)

# Save Generated Outputs
results = pd.DataFrame({
    "Input Text": [sample_review] * 3,
    "Generation Settings": [
        "Beam Search = 1",
        "Beam Search = 4",
        "Beam Search = 6"
    ],
    "Generated Output": [
        summary_1,
        summary_2,
        summary_3
    ]
})

results.to_csv(
    f"24BAD048_T5_Generated_Outputs.csv",
    index=False
)

print("\nGenerated Outputs Saved Successfully!")

print(results)

print("\nPART C COMPLETED SUCCESSFULLY!")

24BAD048 - Jeevaprakash S

Using Device: cuda


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


T5-small Model Loaded Successfully!

Original Input Text:

The movie tells the story of a young man who discovers
his passion for music. Despite facing many challenges,
he works hard to achieve his dreams. The actors deliver
excellent performances and the background music makes
the movie more emotional and enjoyable.


Input Preprocessing Completed!
Input IDs Shape: torch.Size([1, 54])


[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Generated Summary:
the movie tells the story of a young man who discovers his passion for music. the actors deliver excellent performances and the background music makes the movie more emotional and enjoyable.

Original Text:

The film is about a group of friends who travel to a
beautiful island. They experience many adventures and
discover the importance of friendship, teamwork and courage.


Beam Search = 1:
the film is about a group of friends who travel to a beautiful island. they discover the importance of friendship, teamwork and courage.

Beam Search = 4:
a group of friends travel to a beautiful island. they discover the importance of friendship, teamwork and courage.

Beam Search = 6:
film is about a group of friends who travel to a beautiful island. they discover the importance of friendship, teamwork and courage.

Generated Outputs Saved Successfully!
                                          Input Text Generation Settings  \
0  \nThe film is about a group of friends who tra

In [8]:
print("24BAD048 - Jeevaprakash S")


# Import Libraries
import torch
import pandas as pd
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Saved BERT Model
bert_path = f"./24BAD048_BERT_Model"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_path)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    bert_path
).to(device)

bert_model.eval()

print("\nSaved BERT Model Loaded Successfully!")

# Load T5 Model
t5_tokenizer = AutoTokenizer.from_pretrained("t5-small")

t5_model = AutoModelForSeq2SeqLM.from_pretrained(
    "t5-small"
).to(device)

t5_model.eval()

print("T5 Model Loaded Successfully!")

# BERT Prediction Function
def predict_sentiment(text):

    inputs = bert_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = bert_model(**inputs)

        probabilities = torch.softmax(outputs.logits, dim=1)

        prediction = torch.argmax(probabilities, dim=1).item()

    label = "Positive" if prediction == 1 else "Negative"

    return label

# Test Positive and Negative Reviews
test_reviews = [
    "The movie was excellent. I loved every scene.",
    "The acting was terrible and the story was boring.",
    "Amazing direction and wonderful performances.",
    "The movie was a complete waste of time.",
    "An entertaining movie with a beautiful storyline."
]

bert_results = []

for review in test_reviews:

    prediction = predict_sentiment(review)

    print("\nReview:", review)
    print("Predicted Sentiment:", prediction)

    bert_results.append({
        "Review": review,
        "Predicted Sentiment": prediction
    })

bert_results_df = pd.DataFrame(bert_results)

bert_results_df.to_csv(
    f"24BAD048_BERT_Test_Results.csv",
    index=False
)

print("\nBERT Testing Results:")
print(bert_results_df)

# Evaluate BERT Using Held-Out IMDb Test Data
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

test_dataset = dataset["test"].shuffle(seed=42).select(range(1000))

def get_prediction(text):

    inputs = bert_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = bert_model(**inputs)

    return torch.argmax(outputs.logits, dim=1).item()

actual_labels = []
predicted_labels = []

for sample in test_dataset:

    actual_labels.append(sample["label"])

    predicted_labels.append(
        get_prediction(sample["text"])
    )

# Calculate Performance Metrics
accuracy = accuracy_score(actual_labels, predicted_labels)

precision = precision_score(
    actual_labels, predicted_labels, average="weighted"
)

recall = recall_score(
    actual_labels, predicted_labels, average="weighted"
)

f1 = f1_score(
    actual_labels, predicted_labels, average="weighted"
)

print("\nBERT MODEL PERFORMANCE")

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))

# T5 Summary Generation Function
def generate_summary(text, num_beams=4, max_length=80):

    if not text.strip().lower().startswith("summarize:"):
        text = "summarize: " + text

    inputs = t5_tokenizer(
        text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = t5_model.generate(
            inputs["input_ids"],
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=True
        )

    return t5_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

# Test T5 with Different Inputs
sample_texts = [
    """
    The movie follows a brave soldier who saves his
    country from enemies. His courage inspires everyone.
    """,

    """
    A young girl discovers a magical world and learns
    the importance of friendship.
    """,

    """
    A scientist develops new technology that changes
    the world but faces unexpected challenges.
    """
]

t5_results = []

for i, text in enumerate(sample_texts):

    summary = generate_summary(text)

    print("\nInput:", text)
    print("\nGenerated Summary:", summary)

    t5_results.append({
        "Input Text": text.strip(),
        "Generated Summary": summary
    })

t5_results_df = pd.DataFrame(t5_results)

t5_results_df.to_csv(
    f"24BAD048_T5_Test_Results.csv",
    index=False
)

# Compare Generation Settings
comparison_text = """
A young engineer develops an innovative machine
to solve environmental problems. After several
challenges, he successfully demonstrates how
technology can protect nature.
"""

settings = [
    {"num_beams": 1, "max_length": 50},
    {"num_beams": 4, "max_length": 50},
    {"num_beams": 6, "max_length": 100}
]

comparison_results = []

for setting in settings:

    output = generate_summary(
        comparison_text,
        setting["num_beams"],
        setting["max_length"]
    )

    comparison_results.append({
        "Beam Size": setting["num_beams"],
        "Maximum Length": setting["max_length"],
        "Generated Summary": output
    })

comparison_df = pd.DataFrame(comparison_results)

print("\nT5 GENERATION COMPARISON")

print(comparison_df.to_string(index=False))

comparison_df.to_csv(
    f"24BAD048_T5_Comparison.csv",
    index=False
)

# Compare BERT and T5
model_comparison = pd.DataFrame({
    "Feature": [
        "Model",
        "Architecture",
        "Main Task",
        "Input",
        "Output",
        "Application"
    ],

    "BERT": [
        "BERT-base-uncased",
        "Encoder-only",
        "Text Classification",
        "Movie Review",
        "Positive or Negative",
        "Sentiment Analysis"
    ],

    "T5": [
        "T5-small",
        "Encoder-Decoder",
        "Text Generation",
        "Text Input",
        "Generated Summary",
        "Text Summarization"
    ]
})

print("\nBERT VS T5 COMPARISON")
print(model_comparison.to_string(index=False))

model_comparison.to_csv(
    f"24BAD048_Model_Comparison.csv",
    index=False
)

# Final Observations
print("\n====================================")
print("FINAL MODEL ANALYSIS")
print("====================================")

print("\nBERT OBSERVATIONS:")
print("1. BERT classifies reviews into positive and negative.")
print("2. Fine-tuning improves classification performance.")
print("3. Accuracy, precision, recall and F1-score evaluate performance.")

print("\nT5 OBSERVATIONS:")
print("1. T5 generates summaries from input text.")
print("2. Beam search influences generated output.")
print("3. Output quality depends on input and generation settings.")

print("\nOVERALL CONCLUSION:")
print("BERT is suitable for text classification.")
print("T5 is suitable for text generation.")
print("Both models use Transformer architecture.")



24BAD048 - Jeevaprakash S


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Saved BERT Model Loaded Successfully!


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5 Model Loaded Successfully!

Review: The movie was excellent. I loved every scene.
Predicted Sentiment: Positive

Review: The acting was terrible and the story was boring.
Predicted Sentiment: Negative

Review: Amazing direction and wonderful performances.
Predicted Sentiment: Positive

Review: The movie was a complete waste of time.
Predicted Sentiment: Negative

Review: An entertaining movie with a beautiful storyline.
Predicted Sentiment: Positive

BERT Testing Results:
                                              Review Predicted Sentiment
0      The movie was excellent. I loved every scene.            Positive
1  The acting was terrible and the story was boring.            Negative
2      Amazing direction and wonderful performances.            Positive
3            The movie was a complete waste of time.            Negative
4  An entertaining movie with a beautiful storyline.            Positive

BERT MODEL PERFORMANCE
Accuracy : 0.889
Precision: 0.8905
Recall   : 0.889
F1 Sco